In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Environment ready!")

Environment ready!


In [2]:
from google.colab import drive

drive.mount('/content/drive')

print("Google Drive connected!")

Mounted at /content/drive
Google Drive connected!


In [3]:

project_path = "/content/drive/MyDrive/Shaban_Reproduction_Clean"

folders = [
    "data/raw/pm25",
    "data/raw/era5",
    "data/raw/modis",
    "data/processed",
    "results"
]

for folder in folders:
    os.makedirs(os.path.join(project_path, folder), exist_ok=True)

print("Project folders created!")
print(project_path)

Project folders created!
/content/drive/MyDrive/Shaban_Reproduction_Clean


In [4]:
# ==========================================
# 4. OPENAQ SETUP
# ==========================================

import requests
import getpass

OPENAQ_API_KEY = getpass.getpass("Enter your OpenAQ API key: ")

print("OpenAQ key stored for this session.")

Enter your OpenAQ API key: ··········
OpenAQ key stored for this session.


In [5]:
# ==========================================
# 5. TEST OPENAQ
# ==========================================

headers = {
    "X-API-Key": OPENAQ_API_KEY
}

url = "https://api.openaq.org/v3/locations/8118"

response = requests.get(url, headers=headers)

print("Status code:", response.status_code)

if response.status_code == 200:
    print("OpenAQ connection successful!")
else:
    print(response.text[:500])

Status code: 200
OpenAQ connection successful!


In [6]:
# ==========================================
# 6. FIND PM2.5 SENSOR
# ==========================================

data = response.json()
location = data["results"][0]

print("Station:", location["name"])
print("Latitude:", location["coordinates"]["latitude"])
print("Longitude:", location["coordinates"]["longitude"])

for sensor in location["sensors"]:
    if sensor["parameter"]["name"] == "pm25":
        print("PM2.5 Sensor ID:", sensor["id"])
        print("Unit:", sensor["parameter"]["units"])

Station: New Delhi
Latitude: 28.63576
Longitude: 77.22445
PM2.5 Sensor ID: 23534
Unit: µg/m³


In [7]:
# ==========================================
# 7. DOWNLOAD SMALL PM2.5 SAMPLE
# ==========================================

url = "https://api.openaq.org/v3/sensors/23534/hours"

params = {
    "datetime_from": "2024-01-01T00:00:00Z",
    "datetime_to": "2024-01-08T00:00:00Z",
    "limit": 1000
}

r = requests.get(url, headers=headers, params=params)

print("Status:", r.status_code)

pm_data = r.json()["results"]

print("Records downloaded:", len(pm_data))

Status: 200
Records downloaded: 322


In [8]:
# ==========================================
# 8. CLEAN PM2.5 DATA
# ==========================================

pm25_df = pd.DataFrame(pm_data)

pm25_df["datetime"] = pd.to_datetime(
    pm25_df["period"].apply(lambda x: x["datetimeFrom"]["utc"]),
    utc=True
).dt.tz_convert("Asia/Kolkata")

pm25_df["pm25"] = pd.to_numeric(
    pm25_df["value"],
    errors="coerce"
)

# One PM2.5 value per local hour
pm25_hourly = (
    pm25_df
    .set_index("datetime")["pm25"]
    .resample("1h")
    .mean()
    .reset_index()
)

print("Hourly records:", len(pm25_hourly))
print("Valid PM2.5:", pm25_hourly["pm25"].notna().sum())
print(pm25_hourly.head())

Hourly records: 168
Valid PM2.5: 161
                   datetime   pm25
0 2024-01-01 05:00:00+05:30  199.0
1 2024-01-01 06:00:00+05:30  197.0
2 2024-01-01 07:00:00+05:30  194.0
3 2024-01-01 08:00:00+05:30  179.0
4 2024-01-01 09:00:00+05:30  180.0


In [9]:
!pip install -q "cdsapi>=0.7.7"

In [10]:
# ==========================================
# 10. CDS API SETUP
# ==========================================

import os
import getpass
import cdsapi

CDS_API_KEY = getpass.getpass("Enter your CDS Personal Access Token: ")

os.environ["CDSAPI_URL"] = "https://cds.climate.copernicus.eu/api"
os.environ["CDSAPI_KEY"] = CDS_API_KEY

client = cdsapi.Client()

print("CDS connected successfully!")

Enter your CDS Personal Access Token: ··········
CDS connected successfully!


In [12]:
# ==========================================
# 11. DOWNLOAD 7-DAY ERA5 SAMPLE
# ==========================================

era5_path = os.path.join(
    project_path,
    "data/raw/era5/ERA5_Delhi_7days.nc"
)

client.retrieve(
    "reanalysis-era5-single-levels",
    {
        "product_type": ["reanalysis"],
        "variable": [
            "2m_temperature",
            "2m_dewpoint_temperature",
            "10m_u_component_of_wind",
            "10m_v_component_of_wind"
        ],
        "year": ["2024"],
        "month": ["01"],
        "day": ["01", "02", "03", "04", "05", "06", "07"],
        "time": [f"{h:02d}:00" for h in range(24)],
        "data_format": "netcdf",
        "download_format": "unarchived",
        "area": [29.0, 76.8, 28.3, 77.6]
    },
    era5_path
)

print("ERA5 7-day sample downloaded!")
print(era5_path)

2026-09-29 16:17:48,903 INFO Request ID is b86470d3-5ce8-42ad-914c-5db141bdfe27
INFO:ecmwf.datastores.legacy_client:Request ID is b86470d3-5ce8-42ad-914c-5db141bdfe27
2026-09-29 16:17:49,102 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-09-29 16:18:11,376 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-09-29 16:19:05,954 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


8a016761e263737ff6a4757bb824129b.nc:   0%|          | 0.00/73.0k [00:00<?, ?B/s]

ERA5 7-day sample downloaded!
/content/drive/MyDrive/Shaban_Reproduction_Clean/data/raw/era5/ERA5_Delhi_7days.nc


In [13]:
# ==========================================
# 12. CHECK ERA5
# ==========================================

import xarray as xr

era5 = xr.open_dataset(era5_path)

print(era5)
print("\nVariables:", list(era5.data_vars))
print("Hours:", len(era5.valid_time))

<xarray.Dataset> Size: 28kB
Dimensions:     (valid_time: 168, latitude: 3, longitude: 3)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 1kB 2024-01-01 ... 2024-01-07T23:...
  * latitude    (latitude) float64 24B 29.0 28.75 28.5
  * longitude   (longitude) float64 24B 77.0 77.25 77.5
    number      int64 8B ...
    expver      (valid_time) <U4 3kB ...
Data variables:
    t2m         (valid_time, latitude, longitude) float32 6kB ...
    d2m         (valid_time, latitude, longitude) float32 6kB ...
    u10         (valid_time, latitude, longitude) float32 6kB ...
    v10         (valid_time, latitude, longitude) float32 6kB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-09-29T16:18 GRIB to CDM+CF via cfgrib-0.9.1...

V

In [14]:
# ==========================================
# 13. MODIS AOD SAMPLE
# ==========================================

import requests

url = "https://power.larc.nasa.gov/api/temporal/daily/point"

params = {
    "parameters": "AOD_055",
    "community": "AG",
    "longitude": 77.22,
    "latitude": 28.64,
    "start": "20240101",
    "end": "20240107",
    "format": "JSON"
}

r = requests.get(url, params=params)

print("Status:", r.status_code)

if r.status_code == 200:
    aod_data = r.json()["properties"]["parameter"]["AOD_055"]
    print("AOD records:", len(aod_data))
    print(aod_data)
else:
    print(r.text[:500])

Status: 422
{"header":"The POWER Daily API failed to complete your request; please review the errors below and the POWER Docs (https://power.larc.nasa.gov/docs/).","messages":["One of your parameters is incorrect: AOD_055."]}


In [15]:
from google.colab import files

uploaded = files.upload()

Saving MCD19A2.A2024001.h25v05.061.2024004192742.hdf to MCD19A2.A2024001.h25v05.061.2024004192742.hdf


In [16]:
import shutil
import os

src = "/content/MCD19A2.A2024001.h25v05.061.2024004192742.hdf"
dst = "/content/drive/MyDrive/Shaban_Reproduction_Clean/data/raw/modis/MCD19A2.A2024001.h25v05.061.2024004192742.hdf"

shutil.move(src, dst)

print("Satellite file saved:", os.path.exists(dst))

Satellite file saved: True


In [17]:
!pip install -q pyhdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 37.0 MB/s eta 0:00:00


In [18]:
from pyhdf.SD import SD, SDC

hdf_path = "/content/drive/MyDrive/Shaban_Reproduction_Clean/data/raw/modis/MCD19A2.A2024001.h25v05.061.2024004192742.hdf"

hdf = SD(hdf_path, SDC.READ)

datasets = hdf.datasets()

print("Available layers:")
for name in datasets:
    print(name)

Available layers:
Optical_Depth_047
Optical_Depth_055
AOD_Uncertainty
Column_WV
AngstromExp_470-780
AOD_QA
FineModeFraction
Injection_Height
cosSZA
cosVZA
RelAZ
Scattering_Angle
Glint_Angle


In [19]:
aod = hdf.select("Optical_Depth_055")[:]

print("Shape:", aod.shape)
print("Data type:", aod.dtype)
print("Raw min:", aod.min())
print("Raw max:", aod.max())

Shape: (4, 1200, 1200)
Data type: int16
Raw min: -28672
Raw max: 2902


In [20]:
aod_layer = hdf.select("Optical_Depth_055")

print(aod_layer.attributes())

{'long_name': 'AOD at 0.55 micron', 'scale_factor': 0.001, 'add_offset': 0.0, 'unit': 'none', '_FillValue': -28672, 'valid_range': [-100, 6000]}


In [21]:
import numpy as np

aod_raw = hdf.select("Optical_Depth_055")[:]

# Remove invalid/fill values
aod_clean = np.where(
    (aod_raw >= -100) & (aod_raw <= 6000),
    aod_raw * 0.001,
    np.nan
)

print("Valid pixels:", np.sum(~np.isnan(aod_clean)))
print("AOD minimum:", np.nanmin(aod_clean))
print("AOD maximum:", np.nanmax(aod_clean))

Valid pixels: 1724697
AOD minimum: 0.0
AOD maximum: 2.902


In [22]:
print("Latitude:", hdf.select("Optical_Depth_055").attributes().get("valid_range"))
print("Dataset shape:", aod_clean.shape)

Latitude: [-100, 6000]
Dataset shape: (4, 1200, 1200)


In [23]:
from pyproj import Transformer

# Delhi station
lat = 28.63576
lon = 77.22445

# MODIS Sinusoidal projection
transformer = Transformer.from_crs(
    "EPSG:4326",
    "+proj=sinu +R=6371007.181",
    always_xy=True
)

x, y = transformer.transform(lon, lat)

print("MODIS X:", x)
print("MODIS Y:", y)

MODIS X: 7536652.417329605
MODIS Y: 3184154.821590942


In [25]:
from google.colab import files

uploaded = files.upload()

Saving MCD19A2.A2024001.h24v06.061.2024004183207.hdf to MCD19A2.A2024001.h24v06.061.2024004183207.hdf


In [26]:
import glob
import shutil
import os

src = glob.glob("/content/MCD19A2.A2024001.h24v06*.hdf")[0]

dst = "/content/drive/MyDrive/Shaban_Reproduction_Clean/data/raw/modis/" + os.path.basename(src)

shutil.move(src, dst)

print("Saved:", dst)

Saved: /content/drive/MyDrive/Shaban_Reproduction_Clean/data/raw/modis/MCD19A2.A2024001.h24v06.061.2024004183207.hdf


In [27]:
from pyhdf.SD import SD, SDC

hdf_path = "/content/drive/MyDrive/Shaban_Reproduction_Clean/data/raw/modis/MCD19A2.A2024001.h24v06.061.2024004183207.hdf"

hdf_delhi = SD(hdf_path, SDC.READ)

print(hdf_delhi.datasets().keys())

dict_keys(['Optical_Depth_047', 'Optical_Depth_055', 'AOD_Uncertainty', 'Column_WV', 'AngstromExp_470-780', 'AOD_QA', 'FineModeFraction', 'Injection_Height', 'cosSZA', 'cosVZA', 'RelAZ', 'Scattering_Angle', 'Glint_Angle'])


In [28]:
import numpy as np

aod_raw = hdf_delhi.select("Optical_Depth_055")[:]

# Get metadata
attrs = hdf_delhi.select("Optical_Depth_055").attributes()
print(attrs)

# Apply valid range and scale factor
aod_clean = np.where(
    (aod_raw >= -100) & (aod_raw <= 6000),
    aod_raw * 0.001,
    np.nan
)

print("Shape:", aod_clean.shape)
print("Valid pixels:", np.sum(~np.isnan(aod_clean)))
print("AOD minimum:", np.nanmin(aod_clean))
print("AOD maximum:", np.nanmax(aod_clean))

{'long_name': 'AOD at 0.55 micron', 'scale_factor': 0.001, 'add_offset': 0.0, 'unit': 'none', '_FillValue': -28672, 'valid_range': [-100, 6000]}
Shape: (3, 1200, 1200)
Valid pixels: 1734077
AOD minimum: 0.0
AOD maximum: 2.879


In [29]:
from pyproj import Transformer

lat = 28.63576
lon = 77.22445

transformer = Transformer.from_crs(
    "EPSG:4326",
    "+proj=sinu +R=6371007.181",
    always_xy=True
)

x, y = transformer.transform(lon, lat)

print("MODIS X:", x)
print("MODIS Y:", y)

MODIS X: 7536652.417329605
MODIS Y: 3184154.821590942


In [30]:
# MODIS tile h24v06
tile_size = 1111950.519667
pixel_size = tile_size / 1200

# Upper-left corner of h24v06
x0 = (24 - 18) * tile_size
y0 = (9 - 6) * tile_size

# Convert MODIS coordinates to pixel
col = int((x - x0) / pixel_size)
row = int((y0 - y) / pixel_size)

print("Pixel row:", row)
print("Pixel column:", col)

# Check the AOD values at that pixel
print("AOD values:", aod_clean[:, row, col])

Pixel row: 163
Pixel column: 933
AOD values: [  nan 1.262   nan]


In [31]:
# Satellite AOD value extracted above
aod_550 = 1.262

# First valid PM2.5 value from our Jan 1 sample
pm25_value = pm25_hourly.loc[
    pm25_hourly["pm25"].notna(), "pm25"
].iloc[0]

# Create a simple combined row
combined = pd.DataFrame({
    "datetime": [pm25_hourly.loc[
        pm25_hourly["pm25"].notna(), "datetime"
    ].iloc[0]],
    "latitude": [28.63576],
    "longitude": [77.22445],
    "pm25": [pm25_value],
    "aod_550": [aod_550]
})

print(combined)

                   datetime  latitude  longitude   pm25  aod_550
0 2024-01-01 05:00:00+05:30  28.63576   77.22445  199.0    1.262


In [32]:
import xarray as xr

era5 = xr.open_dataset(
    "/content/drive/MyDrive/Shaban_Reproduction_Clean/data/raw/era5/ERA5_Delhi_7days.nc"
)

print(era5)

<xarray.Dataset> Size: 28kB
Dimensions:     (valid_time: 168, latitude: 3, longitude: 3)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 1kB 2024-01-01 ... 2024-01-07T23:...
  * latitude    (latitude) float64 24B 29.0 28.75 28.5
  * longitude   (longitude) float64 24B 77.0 77.25 77.5
    number      int64 8B ...
    expver      (valid_time) <U4 3kB ...
Data variables:
    t2m         (valid_time, latitude, longitude) float32 6kB ...
    d2m         (valid_time, latitude, longitude) float32 6kB ...
    u10         (valid_time, latitude, longitude) float32 6kB ...
    v10         (valid_time, latitude, longitude) float32 6kB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-09-29T16:18 GRIB to CDM+CF via cfgrib-0.9.1...


In [33]:
# Convert 05:00 IST to UTC
target_time = pd.Timestamp("2024-01-01 05:00:00", tz="Asia/Kolkata").tz_convert("UTC").tz_localize(None)

# Select nearest ERA5 grid point and time
era5_point = era5.sel(
    valid_time=target_time,
    latitude=28.63576,
    longitude=77.22445,
    method="nearest"
)

print("ERA5 time:", era5_point.valid_time.values)
print("ERA5 latitude:", era5_point.latitude.values)
print("ERA5 longitude:", era5_point.longitude.values)

print("Temperature (K):", era5_point.t2m.values)
print("Dew point (K):", era5_point.d2m.values)
print("U wind (m/s):", era5_point.u10.values)
print("V wind (m/s):", era5_point.v10.values)

ERA5 time: 2024-01-01T00:00:00.000000000
ERA5 latitude: 28.75
ERA5 longitude: 77.25
Temperature (K): 282.146
Dew point (K): 281.2126
U wind (m/s): 0.55740356
V wind (m/s): -1.7991486


In [34]:
combined["temperature_K"] = 282.146
combined["dewpoint_K"] = 281.2126
combined["u10"] = 0.55740356
combined["v10"] = -1.7991486

print(combined)

                   datetime  latitude  longitude   pm25  aod_550  \
0 2024-01-01 05:00:00+05:30  28.63576   77.22445  199.0    1.262   

   temperature_K  dewpoint_K       u10       v10  
0        282.146    281.2126  0.557404 -1.799149  


In [35]:
from google.colab import files

uploaded = files.upload()

Saving MCD19A2.A2024007.h24v06.061.2024009043118.hdf to MCD19A2.A2024007.h24v06.061.2024009043118.hdf
Saving MCD19A2.A2024006.h24v06.061.2024009024449.hdf to MCD19A2.A2024006.h24v06.061.2024009024449.hdf
Saving MCD19A2.A2024005.h24v06.061.2024009001728.hdf to MCD19A2.A2024005.h24v06.061.2024009001728.hdf
Saving MCD19A2.A2024004.h24v06.061.2024006003425.hdf to MCD19A2.A2024004.h24v06.061.2024006003425.hdf
Saving MCD19A2.A2024003.h24v06.061.2024005152156.hdf to MCD19A2.A2024003.h24v06.061.2024005152156.hdf
Saving MCD19A2.A2024002.h24v06.061.2024005024035.hdf to MCD19A2.A2024002.h24v06.061.2024005024035.hdf


In [36]:
import glob
import os
import re
import numpy as np
import pandas as pd
from pyhdf.SD import SD, SDC

# Delhi station
lat = 28.63576
lon = 77.22445

# MODIS h24v06 pixel
row = 163
col = 933

# Find uploaded h24v06 files
files = sorted(glob.glob("/content/MCD19A2.A2024*.h24v06*.hdf"))

print("Files found:", len(files))

records = []

for file in files:
    hdf = SD(file, SDC.READ)

    raw = hdf.select("Optical_Depth_055")[:]

    # Apply valid range + scale factor
    clean = np.where(
        (raw >= -100) & (raw <= 6000),
        raw * 0.001,
        np.nan
    )

    # AOD at Delhi pixel
    values = clean[:, row, col]
    aod = np.nanmean(values)

    # Extract date from filename
    match = re.search(r"A(\d{7})", os.path.basename(file))
    date_code = match.group(1) if match else "unknown"

    records.append({
        "date_code": date_code,
        "aod_550": aod
    })

    hdf.end()

aod_df = pd.DataFrame(records).sort_values("date_code").reset_index(drop=True)

print(aod_df)

Files found: 6


/tmp/ipykernel_2444/523182640.py:37: RuntimeWarning: Mean of empty slice
  aod = np.nanmean(values)


  date_code  aod_550
0   2024002    0.727
1   2024003      NaN
2   2024004      NaN
3   2024005      NaN
4   2024006      NaN
5   2024007    2.169


In [37]:
# Check a 5x5 pixel neighborhood around Delhi
for file in files:
    hdf = SD(file, SDC.READ)

    raw = hdf.select("Optical_Depth_055")[:]

    clean = np.where(
        (raw >= -100) & (raw <= 6000),
        raw * 0.001,
        np.nan
    )

    # 5x5 neighborhood around Delhi pixel
    neighborhood = clean[:, row-2:row+3, col-2:col+3]

    valid = neighborhood[~np.isnan(neighborhood)]

    date_code = re.search(
        r"A(\d{7})",
        os.path.basename(file)
    ).group(1)

    print(
        date_code,
        "valid nearby pixels:",
        len(valid),
        "mean AOD:",
        np.nanmean(valid) if len(valid) > 0 else np.nan
    )

    hdf.end()

2024002 valid nearby pixels: 25 mean AOD: 0.74492
2024003 valid nearby pixels: 0 mean AOD: nan
2024004 valid nearby pixels: 0 mean AOD: nan
2024005 valid nearby pixels: 0 mean AOD: nan
2024006 valid nearby pixels: 0 mean AOD: nan
2024007 valid nearby pixels: 25 mean AOD: 2.15596


In [38]:
aod_df.to_csv(
    "/content/drive/MyDrive/Shaban_Reproduction_Clean/data/processed/Delhi_AOD_Jan1_7.csv",
    index=False
)

print("Saved:")
print(aod_df)

Saved:
  date_code  aod_550
0   2024002    0.727
1   2024003      NaN
2   2024004      NaN
3   2024005      NaN
4   2024006      NaN
5   2024007    2.169


In [39]:
print(pm25_hourly[["datetime", "pm25"]].dropna().head(10))
print("\nTotal valid PM2.5 observations:", pm25_hourly["pm25"].notna().sum())

                    datetime   pm25
0  2024-01-01 05:00:00+05:30  199.0
1  2024-01-01 06:00:00+05:30  197.0
2  2024-01-01 07:00:00+05:30  194.0
3  2024-01-01 08:00:00+05:30  179.0
4  2024-01-01 09:00:00+05:30  180.0
5  2024-01-01 10:00:00+05:30  182.0
7  2024-01-01 12:00:00+05:30  196.0
8  2024-01-01 13:00:00+05:30  232.0
9  2024-01-01 14:00:00+05:30  252.0
10 2024-01-01 15:00:00+05:30  236.0

Total valid PM2.5 observations: 161


In [40]:
# Daily mean PM2.5
pm25_daily = (
    pm25_hourly
    .dropna(subset=["pm25"])
    .assign(date=lambda x: x["datetime"].dt.date)
    .groupby("date", as_index=False)["pm25"]
    .mean()
)

# Convert satellite date codes to normal dates
aod_df["date"] = pd.to_datetime(
    aod_df["date_code"].str[0:4] + "-" +
    aod_df["date_code"].str[4:7],
    format="%Y-%j"
).dt.date

# Merge daily PM2.5 with satellite AOD
combined_daily = pd.merge(
    pm25_daily,
    aod_df[["date", "aod_550"]],
    on="date",
    how="left"
)

print(combined_daily)

         date        pm25  aod_550
0  2024-01-01  217.500000      NaN
1  2024-01-02  205.391304    0.727
2  2024-01-03  222.652174      NaN
3  2024-01-04  257.739130      NaN
4  2024-01-05  202.173913      NaN
5  2024-01-06  197.391304      NaN
6  2024-01-07  193.608696    2.169
7  2024-01-08  186.600000      NaN


In [41]:
# Add the verified Jan 1 satellite AOD
jan1 = pd.DataFrame({
    "date": [pd.to_datetime("2024-01-01").date()],
    "aod_550": [1.262]
})

aod_all = pd.concat([
    jan1,
    aod_df[["date", "aod_550"]]
], ignore_index=True)

# Merge again
combined_daily = pd.merge(
    pm25_daily,
    aod_all,
    on="date",
    how="left"
)

# Keep only days with real satellite AOD
ml_data = combined_daily.dropna(subset=["aod_550"]).reset_index(drop=True)

print(ml_data)
print("\nNumber of usable days:", len(ml_data))

         date        pm25  aod_550
0  2024-01-01  217.500000    1.262
1  2024-01-02  205.391304    0.727
2  2024-01-07  193.608696    2.169

Number of usable days: 3


In [42]:
import numpy as np
import pandas as pd

# Our three real satellite-observation days
demo = ml_data.copy()

# Add the ERA5 values we already extracted for demonstration
demo["temperature_K"] = 282.146
demo["dewpoint_K"] = 281.2126
demo["u10"] = 0.55740356
demo["v10"] = -1.7991486

features = [
    "aod_550",
    "temperature_K",
    "dewpoint_K",
    "u10",
    "v10"
]

X_demo = demo[features].values.astype("float32")
y_demo = demo["pm25"].values.astype("float32")

print("X shape:", X_demo.shape)
print("y shape:", y_demo.shape)
print(X_demo)
print(y_demo)

X shape: (3, 5)
y shape: (3,)
[[  1.262      282.146      281.2126       0.55740356  -1.7991486 ]
 [  0.727      282.146      281.2126       0.55740356  -1.7991486 ]
 [  2.169      282.146      281.2126       0.55740356  -1.7991486 ]]
[217.5     205.39131 193.60869]


In [43]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, LSTM, Dense

# Create sequence:
# [Day 1, Day 2] -> Day 3
X_seq = np.array([
    X_demo[:2]
], dtype="float32")

y_seq = np.array([
    y_demo[2]
], dtype="float32")

print("Input shape:", X_seq.shape)
print("Target shape:", y_seq.shape)

# CNN-LSTM model
model = Sequential([
    Conv1D(
        filters=32,
        kernel_size=2,
        activation="relu",
        input_shape=(2, 5)
    ),
    LSTM(32),
    Dense(16, activation="relu"),
    Dense(1)
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()

Input shape: (1, 2, 5)
Target shape: (1,)


/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 1, 32)          │           352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,217 (36.00 KB)

 Trainable params: 9,217 (36.00 KB)

 Non-trainable params: 0 (0.00 B)

In [44]:
history = model.fit(
    X_seq,
    y_seq,
    epochs=20,
    verbose=1
)

print("\nTraining completed.")

Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - loss: 37505.8984 - mae: 193.6644
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step - loss: 37446.5547 - mae: 193.5111
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step - loss: 37429.6914 - mae: 193.4675
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - loss: 37414.6289 - mae: 193.4286
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - loss: 37388.0586 - mae: 193.3599
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step - loss: 37361.4453 - mae: 193.2911
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step - loss: 37345.0273 - mae: 193.2486
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - loss: 37314.5820 - mae: 193.1698
Epoch 9/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - loss: 37287.9961 - mae: 193.1010
Epoch 10/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - loss: 37272.9688 - mae: 193.0621
Epoch 11/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step - loss: 37266.1172 - mae: 193.0443
Epoch 12/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - loss: 37259.3477 - 

In [45]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Predict Day 3 PM2.5
y_pred = model.predict(X_seq, verbose=0).flatten()

# Actual value
y_actual = y_seq

# Evaluation
mae = mean_absolute_error(y_actual, y_pred)
rmse = np.sqrt(mean_squared_error(y_actual, y_pred))

print("Actual PM2.5    :", y_actual)
print("Predicted PM2.5 :", y_pred)

print("\nMAE  :", mae)
print("RMSE :", rmse)

Actual PM2.5    : [193.60869]
Predicted PM2.5 : [0.7496328]

MAE  : 192.8590545654297
RMSE : 192.85905029645355
